#NDS Event Matching


It matches each event from `nds_events.xlsx` to the most likely operation entry in `well_reports.sqlite`.




In [1]:
import re
import sqlite3
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 180)

## 1. Locate input files

Upload these two files before running the notebook:

- `well_reports.sqlite`
- `nds_events.xlsx`

The helper searches the current folder, `/content`, and `/mnt/data`, so it works in Colab and locally.

In [2]:
def find_file(patterns, search_roots=None):
    if search_roots is None:
        search_roots = [Path.cwd(), Path('/content'), Path('/mnt/data')]

    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            matches = sorted(root.glob(pattern))
            if matches:
                return matches[0]

    raise FileNotFoundError(f'Could not find file matching: {patterns}')


try:
    SQLITE_PATH = find_file(['well_reports*.sqlite', '*.sqlite'])
    NDS_PATH = find_file(['nds_events*.xlsx'])
except FileNotFoundError:
    # Colab fallback: upload files only if they are not already available.
    from google.colab import files
    files.upload()
    SQLITE_PATH = find_file(['well_reports*.sqlite', '*.sqlite'])
    NDS_PATH = find_file(['nds_events*.xlsx'])

print('SQLite:', SQLITE_PATH)
print('NDS events:', NDS_PATH)

SQLite: /mnt/data/well_reports.sqlite
NDS events: /mnt/data/nds_events(5).xlsx


## 2. Load NDS events

Only `Well` and `Event` are required for the matching task.

In [3]:
nds = pd.read_excel(NDS_PATH)
nds = nds[['Well', 'Event']].copy()
nds['event_id'] = np.arange(1, len(nds) + 1)
nds['Well'] = nds['Well'].astype(str).str.strip()
nds['Event'] = nds['Event'].astype(str).str.strip()

print('NDS rows:', len(nds))
nds[['event_id', 'Well', 'Event']]

NDS rows: 4


,event_id,Well,Event
0,1,15/9-F-10,inclination angle was higher than expected. Reduced inclination to to 0.16 deg
1,2,15/9-F-11,tight hole event was ecountered while drilling interval at 958 m
2,3,15/9-F-12,"Excessive clay amount ccumulation is observed in BHA while POOH 26"""" BHA"
3,4,15/9-F-13,"while RIH with 20"" casing there was a differential stuck. The mud was replaced to seawater and 300 m3 of seawater was pumped, Set down weight was increased and string came free"


## 3. Text cleaning and operation candidates

The searchable operation text combines:

`main_activity + sub_activity + state + remark`

This is intentional because the Task 1 database sometimes stores the long operation description in `state` instead of `remark`. Combining all text fields prevents losing the real operation excerpt.

In [4]:
STOPWORDS = {
    'the', 'and', 'was', 'were', 'while', 'with', 'there', 'from', 'that',
    'this', 'into', 'for', 'to', 'of', 'in', 'on', 'at', 'as', 'is', 'are',
    'a', 'an', 'be', 'by', 'it', 'or', 'if', 'then', 'than', 'had', 'has', 'have'
}

GENERIC_DRILLING_TERMS = {
    'drilling', 'drill', 'drilled', 'operation', 'operations', 'report', 'section',
    'well', 'hole', 'interval', 'time', 'from', 'into', 'with', 'while', 'during',
    'run', 'pull', 'out', 'pooh', 'rih'
}

CRITICAL_TOKEN_WEIGHTS = {
    'inclination': 3.0, 'angle': 1.5, 'reduce': 2.0, 'reduced': 2.0, 'deg': 1.5,
    'tight': 3.0, 'encountered': 2.0, 'overpull': 2.0,
    'clay': 4.0, 'covered': 2.0, 'stabilizer': 2.0, 'stabilizers': 2.0,
    'accumulation': 2.0, 'bha': 1.0,
    'differential': 4.0, 'stuck': 4.0, 'seawater': 4.0, 'pump': 2.5,
    'pumped': 2.5, 'free': 2.5, 'string': 2.0, 'weight': 2.0,
    'set': 1.5, 'down': 1.5, 'replaced': 2.0, 'casing': 1.5,
}

DOMAIN_PHRASES = [
    'inclination', 'tight hole', 'tight', 'overpull', 'clay', 'covered with clay',
    'differential stuck', 'differential', 'stuck', 'seawater', 'set down',
    'string came free', 'free string', '20" casing', '20 casing', '26" bha', '26 bha'
]


def clean_text(x):
    if pd.isna(x):
        return ''

    x = str(x).lower().replace(',', '.')

    replacements = {
        'iclinatio n': 'inclination',
        'incl ination': 'inclination',
        'ecountered': 'encountered',
        'encounterd': 'encountered',
        'thight': 'tight',
        'drlling': 'drilling',
        'comming': 'coming',
        'ccumulation': 'accumulation',
        'aaccumulation': 'accumulation',
        'reduced': 'reduce',
        'pumped': 'pump',
        'pooh': 'pull out of hole pooh',
        'rih': 'run in hole rih',
    }
    for wrong, right in replacements.items():
        x = x.replace(wrong, right)

    x = re.sub(r'"+', '"', x)
    x = re.sub(r'[^a-z0-9\.\-/"\s]+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def normalize_well(x):
    return str(x).strip().upper()


def safe_join(*parts):
    return ' '.join(str(p).strip() for p in parts if pd.notna(p) and str(p).strip())


def keyword_tokens(text):
    tokens = set()
    for token in clean_text(text).split():
        token = token.strip('."')
        if len(token) <= 2:
            continue
        if token in STOPWORDS:
            continue
        if re.fullmatch(r'\d+(?:\.\d+)?', token):
            continue
        tokens.add(token)
    return tokens


def number_tokens(text):
    return re.findall(r'\d+(?:\.\d+)?', clean_text(text))

In [5]:
def load_operation_candidates(sqlite_path):
    query = """
        SELECT
            r.id AS report_id,
            r.filename,
            r.file_path,
            r.wellbore_id,
            r.report_number,
            r.period_start,
            r.period_end,
            o.id AS operation_id,
            o.start_time,
            o.end_time,
            o.end_depth_mmd,
            o.main_activity,
            o.sub_activity,
            o.state,
            o.remark
        FROM operations o
        JOIN reports r ON o.report_id = r.id
    """

    with sqlite3.connect(sqlite_path) as con:
        candidates = pd.read_sql_query(query, con)

    candidates['candidate_text'] = candidates.apply(
        lambda row: safe_join(row['main_activity'], row['sub_activity'], row['state'], row['remark']),
        axis=1
    )
    candidates['candidate_text_clean'] = candidates['candidate_text'].apply(clean_text)
    candidates['well_key'] = candidates['wellbore_id'].apply(normalize_well)
    candidates = candidates[candidates['candidate_text_clean'].str.len() > 0].copy()
    return candidates


candidates = load_operation_candidates(SQLITE_PATH)

print('Operation candidates:', len(candidates))
print('Unique wells:', candidates['wellbore_id'].nunique())

candidates[['wellbore_id', 'filename', 'operation_id', 'candidate_text']].head()

Operation candidates: 10983
Unique wells: 14


,wellbore_id,filename,operation_id,candidate_text
0,15/9-F-12,15_9_F_12_2007_09_11.pdf,1,"interruption -- other ok Worked to release overshot lock ring. Used C-spanner, chain hoist. Rigged up eagle clamp and sheave. Pulled on lock ring C-spanner with tugger. Lock ri..."
1,15/9-F-12,15_9_F_12_2007_09_11.pdf,2,"drilling -- bop/wellhea d equipment ok Removed tools and slings from BOP. Attached sling to overshot and lifted same to drillfloor with tugger. Installed masterbushing, bushing..."
2,15/9-F-12,15_9_F_12_2007_09_11.pdf,3,drilling -- bop/wellhea d equipment ok Prepared for skidding BOP. Released NT-2 connector and removed obstacles in moonpool. Cleared and tidied drillfloor.
3,15/9-F-12,15_9_F_12_2007_09_11.pdf,4,drilling -- bop/wellhea d equipment ok Continued nippling down BOP. Prepared for skidding BOP. Got problems with hydraulics on BOP trolley.
4,15/9-F-12,15_9_F_12_2007_09_11.pdf,5,interruption -- other ok Troubleshot problems with BOP trolley.


## 4. Similarity methods



In [6]:
LOW_CONFIDENCE_THRESHOLD = 0.25
HIGH_CONFIDENCE_THRESHOLD = 0.35
LOW_MARGIN_THRESHOLD = 0.03
F10_OVERRIDE_MARGIN = 0.05
TOP_N = 3
TASK_SOURCE_WELL = '15/9-F-10'

SCORE_COLUMNS = [
    'word_tfidf_score',
    'char_tfidf_score',
    'critical_token_score',
    'domain_phrase_score',
    'number_overlap_score',
    'token_overlap_score',
    'fuzzy_score',
    'hybrid_score',
]

METHOD_LABELS = {
    'word_tfidf_score': 'Word TF-IDF',
    'char_tfidf_score': 'Character TF-IDF',
    'critical_token_score': 'Critical token score',
    'domain_phrase_score': 'Domain phrase score',
    'number_overlap_score': 'Number overlap',
    'token_overlap_score': 'Token overlap',
    'fuzzy_score': 'Fuzzy similarity',
    'hybrid_score': 'Hybrid final score',
}


def tfidf_scores(query_text, candidate_texts, mode):
    docs = [query_text] + list(candidate_texts)

    if mode == 'word':
        vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', min_df=1)
    elif mode == 'char':
        vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1)
    else:
        raise ValueError("mode must be 'word' or 'char'")

    matrix = vectorizer.fit_transform(docs)
    return cosine_similarity(matrix[0:1], matrix[1:]).ravel()


def token_overlap_score(query_text, candidate_text):
    q = keyword_tokens(query_text)
    c = keyword_tokens(candidate_text)
    if not q:
        return 0.0
    return len(q & c) / len(q)


def number_overlap_score(query_text, candidate_text):
    q_nums = number_tokens(query_text)
    if not q_nums:
        return 0.0
    c = clean_text(candidate_text)
    return sum(1 for n in q_nums if n in c) / len(q_nums)


def domain_phrase_score(query_text, candidate_text):
    q_clean = clean_text(query_text)
    c_clean = clean_text(candidate_text)
    q_phrases = [phrase for phrase in DOMAIN_PHRASES if phrase in q_clean]
    if not q_phrases:
        return 0.0
    return sum(1 for phrase in q_phrases if phrase in c_clean) / len(q_phrases)


def critical_token_score(query_text, candidate_text, pool_vocabulary=None):
    q_tokens = keyword_tokens(query_text)
    c_tokens = keyword_tokens(candidate_text)

    q_critical = []
    for token in q_tokens:
        if pool_vocabulary is not None and token not in pool_vocabulary:
            continue
        if token in CRITICAL_TOKEN_WEIGHTS or token not in GENERIC_DRILLING_TERMS:
            q_critical.append(token)

    if not q_critical:
        q_critical = [t for t in q_tokens if pool_vocabulary is None or t in pool_vocabulary]

    if not q_critical:
        return 0.0

    denom = sum(CRITICAL_TOKEN_WEIGHTS.get(t, 1.0) for t in q_critical)
    numer = sum(CRITICAL_TOKEN_WEIGHTS.get(t, 1.0) for t in q_critical if t in c_tokens)
    return numer / denom if denom else 0.0


def fuzzy_score(query_text, candidate_text):
    return SequenceMatcher(None, clean_text(query_text), clean_text(candidate_text)).ratio()


def score_pool_for_event(event_text, pool):
    event_clean = clean_text(event_text)
    scored = pool.copy().reset_index(drop=True)
    texts = scored['candidate_text_clean'].tolist()
    pool_vocabulary = set().union(*(keyword_tokens(text) for text in texts)) if texts else set()

    scored['word_tfidf_score'] = tfidf_scores(event_clean, texts, mode='word')
    scored['char_tfidf_score'] = tfidf_scores(event_clean, texts, mode='char')
    scored['critical_token_score'] = [critical_token_score(event_clean, text, pool_vocabulary) for text in texts]
    scored['domain_phrase_score'] = [domain_phrase_score(event_clean, text) for text in texts]
    scored['number_overlap_score'] = [number_overlap_score(event_clean, text) for text in texts]
    scored['token_overlap_score'] = [token_overlap_score(event_clean, text) for text in texts]
    scored['fuzzy_score'] = [fuzzy_score(event_clean, text) for text in texts]

    scored['hybrid_score'] = (
        0.18 * scored['word_tfidf_score']
        + 0.18 * scored['char_tfidf_score']
        + 0.28 * scored['critical_token_score']
        + 0.16 * scored['domain_phrase_score']
        + 0.10 * scored['number_overlap_score']
        + 0.07 * scored['token_overlap_score']
        + 0.03 * scored['fuzzy_score']
    )

    return scored.sort_values('hybrid_score', ascending=False).reset_index(drop=True)


def confidence_label(score):
    if score >= HIGH_CONFIDENCE_THRESHOLD:
        return 'HIGH'
    if score >= LOW_CONFIDENCE_THRESHOLD:
        return 'MEDIUM'
    return 'LOW' 

## 5. Strict matching plus 15/9-F-10 fallback

For each NDS event, the notebook scores two pools:

1. `STRICT_WELL`: operation rows where `wellbore_id == nds_events['Well']`
2. `TASK_WELL_FALLBACK`: operation rows from `15/9-F-10`

The final match is selected from the better/valid pool, but both pools are saved for review.

In [7]:
def format_top_rows(scored, nds_row, scope_label, top_n=TOP_N):
    # Remove duplicate text excerpts from the same PDF so the top candidates are easier to review.
    unique_scored = scored.drop_duplicates(subset=['filename', 'candidate_text_clean']).copy()
    top = unique_scored.head(top_n).copy()

    top.insert(0, 'rank', np.arange(1, len(top) + 1))
    top.insert(0, 'scope', scope_label)
    top.insert(0, 'event', nds_row['Event'])
    top.insert(0, 'well_from_nds', nds_row['Well'])
    top.insert(0, 'event_id', int(nds_row['event_id']))

    output_cols = [
        'event_id', 'well_from_nds', 'event', 'scope', 'rank', 'filename', 'wellbore_id',
        'report_number', 'period_start', 'period_end', 'operation_id',
        'start_time', 'end_time', 'end_depth_mmd', *SCORE_COLUMNS, 'candidate_text'
    ]

    top = top[output_cols].rename(columns={
        'filename': 'matched_pdf',
        'wellbore_id': 'matched_well_in_database',
        'start_time': 'operation_start_time',
        'end_time': 'operation_end_time',
        'end_depth_mmd': 'operation_depth_mmd',
        'candidate_text': 'matched_excerpt',
    })

    for col in SCORE_COLUMNS:
        top[col] = top[col].round(4)

    return top


def top1_from_top_rows(top_rows):
    if top_rows is None or len(top_rows) == 0:
        return None
    return top_rows.iloc[0].to_dict()


def choose_scope(strict_top1, task_top1, global_top1, nds_well):
    nds_is_task_well = normalize_well(nds_well) == normalize_well(TASK_SOURCE_WELL)

    if strict_top1 is None and task_top1 is not None:
        return 'TASK_WELL_FALLBACK', 'Strict NDS well was not found in the database; used the task-source well 15/9-F-10.'

    if strict_top1 is None and global_top1 is not None:
        return 'GLOBAL_FALLBACK', 'Strict NDS well and task-source well were unavailable; used global search.'

    if task_top1 is None:
        return 'STRICT_WELL', 'Task-source well search was unavailable; used strict NDS well match.'

    if nds_is_task_well:
        return 'STRICT_WELL', 'NDS well already equals the task-source well.'

    strict_score = float(strict_top1['hybrid_score'])
    task_score = float(task_top1['hybrid_score'])

    if strict_score < LOW_CONFIDENCE_THRESHOLD and task_score > strict_score:
        return 'TASK_WELL_FALLBACK', 'Strict match was low-confidence and 15/9-F-10 scored higher.'

    if task_score >= LOW_CONFIDENCE_THRESHOLD and task_score >= strict_score + F10_OVERRIDE_MARGIN:
        return 'TASK_WELL_FALLBACK', '15/9-F-10 candidate was clearly stronger than the strict NDS-well candidate.'

    return 'STRICT_WELL', 'Strict NDS-well candidate was kept because it was competitive.'


def no_match_row(nds_row, status):
    row = {
        'event_id': int(nds_row['event_id']),
        'well_from_nds': nds_row['Well'],
        'event': nds_row['Event'],
        'status': status,
        'final_scope': None,
        'selection_reason': None,
        'candidate_pool_size': 0,
        'matched_pdf': None,
        'matched_well_in_database': None,
        'report_number': None,
        'period_start': None,
        'period_end': None,
        'operation_id': None,
        'operation_start_time': None,
        'operation_end_time': None,
        'operation_depth_mmd': None,
        'hybrid_gap_to_second': None,
        'confidence_label': 'NO_MATCH',
        'review_flag': True,
        'matched_excerpt': None,
    }
    row.update({col: np.nan for col in SCORE_COLUMNS})
    return row

In [8]:
def build_matches(nds, candidates):
    best_rows = []
    top_frames = []
    scored_by_event_scope = {}
    final_scored_by_event = {}

    task_key = normalize_well(TASK_SOURCE_WELL)

    for _, nds_row in nds.iterrows():
        event_id = int(nds_row['event_id'])
        strict_key = normalize_well(nds_row['Well'])

        scopes = {
            'STRICT_WELL': candidates[candidates['well_key'] == strict_key].copy(),
            'TASK_WELL_FALLBACK': candidates[candidates['well_key'] == task_key].copy(),
        }

        # This should rarely be needed, but it prevents the notebook from failing if inputs change.
        if scopes['STRICT_WELL'].empty and scopes['TASK_WELL_FALLBACK'].empty:
            scopes['GLOBAL_FALLBACK'] = candidates.copy()

        top_by_scope = {}
        scored_by_scope = {}

        for scope_label, pool in scopes.items():
            if pool.empty:
                continue

            scored = score_pool_for_event(nds_row['Event'], pool)
            scored_by_scope[scope_label] = scored
            scored_by_event_scope[(event_id, scope_label)] = scored

            top_rows = format_top_rows(scored, nds_row, scope_label, top_n=TOP_N)
            top_by_scope[scope_label] = top_rows
            top_frames.append(top_rows)

        strict_top1 = top1_from_top_rows(top_by_scope.get('STRICT_WELL'))
        task_top1 = top1_from_top_rows(top_by_scope.get('TASK_WELL_FALLBACK'))
        global_top1 = top1_from_top_rows(top_by_scope.get('GLOBAL_FALLBACK'))

        if not top_by_scope:
            best_rows.append(no_match_row(nds_row, 'NO_MATCH_NO_OPERATION_CANDIDATES'))
            continue

        final_scope, selection_reason = choose_scope(strict_top1, task_top1, global_top1, nds_row['Well'])
        final_top_rows = top_by_scope[final_scope]
        final_scored = scored_by_scope[final_scope]
        final_scored_by_event[event_id] = final_scored

        best = final_top_rows.iloc[0].to_dict()
        second_score = float(final_top_rows.iloc[1]['hybrid_score']) if len(final_top_rows) > 1 else 0.0
        gap = round(float(best['hybrid_score']) - second_score, 4)
        pool_size = len(scopes[final_scope]) if final_scope in scopes else len(candidates)

        best['final_scope'] = final_scope
        best['selection_reason'] = selection_reason
        best['candidate_pool_size'] = pool_size
        best['hybrid_gap_to_second'] = gap
        best['confidence_label'] = confidence_label(float(best['hybrid_score']))
        best['review_flag'] = bool(best['hybrid_score'] < LOW_CONFIDENCE_THRESHOLD or gap < LOW_MARGIN_THRESHOLD)
        best['status'] = 'MATCHED_REVIEW' if best['review_flag'] else 'MATCHED'

        best['strict_candidate_count'] = len(scopes['STRICT_WELL'])
        best['task_well_candidate_count'] = len(scopes['TASK_WELL_FALLBACK'])
        best['strict_best_pdf'] = strict_top1['matched_pdf'] if strict_top1 else None
        best['strict_best_score'] = strict_top1['hybrid_score'] if strict_top1 else np.nan
        best['strict_best_excerpt'] = strict_top1['matched_excerpt'] if strict_top1 else None
        best['task_well_best_pdf'] = task_top1['matched_pdf'] if task_top1 else None
        best['task_well_best_score'] = task_top1['hybrid_score'] if task_top1 else np.nan
        best['task_well_best_excerpt'] = task_top1['matched_excerpt'] if task_top1 else None

        best_rows.append(best)

    best_matches = pd.DataFrame(best_rows)
    top_candidates = pd.concat(top_frames, ignore_index=True) if top_frames else pd.DataFrame()
    return best_matches, top_candidates, scored_by_event_scope, final_scored_by_event


best_matches, top_candidates, scored_by_event_scope, final_scored_by_event = build_matches(nds, candidates)

## 6. Final Task 2 results

This is the final one-row-per-event output required by the task.

In [9]:
best_matches[[
    'event_id',
    'well_from_nds',
    'final_scope',
    'matched_pdf',
    'matched_well_in_database',
    'operation_start_time',
    'operation_end_time',
    'hybrid_score',
    'confidence_label',
    'selection_reason',
    'matched_excerpt'
]]

,event_id,well_from_nds,final_scope,matched_pdf,matched_well_in_database,operation_start_time,operation_end_time,hybrid_score,confidence_label,selection_reason,matched_excerpt
0,1,15/9-F-10,STRICT_WELL,15_9_F_10_2009_04_12.pdf,15/9-F-10,09:00,11:15,0.6956,HIGH,NDS well already equals the task-source well.,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
1,2,15/9-F-11,TASK_WELL_FALLBACK,15_9_F_10_2009_04_15.pdf,15/9-F-10,20:30,00:00,0.6752,HIGH,15/9-F-10 candidate was clearly stronger than the strict NDS-well candidate.,"drilling -- trip ok Encounterd thight hole at 958 m MD, took 20-30 MT overpull. Worked string with rotation/flow as limited by torque and overpull. Parameters : Flow 200-800 lp..."
2,3,15/9-F-12,TASK_WELL_FALLBACK,15_9_F_10_2009_04_16.pdf,15/9-F-10,07:00,09:45,0.4403,HIGH,15/9-F-10 candidate was clearly stronger than the strict NDS-well candidate.,"drilling -- trip ok Broke and racked back stand of 8,25"" DC and stand of 9,25"" NMDC. Broke connection between gMWD and MWD, retrieved gyro insert, re-made connection. Attempted..."
3,4,15/9-F-13,TASK_WELL_FALLBACK,15_9_F_10_2009_04_18.pdf,15/9-F-10,07:30,08:30,0.5896,HIGH,Strict NDS well was not found in the database; used the task-source well 15/9-F-10.,"interruptio n -- fish ok Attempted to RIH with 20"" casing / 18 3/4"" WH - negative. Worked to free pipe with alternating up/down weights, 50 MT set down / 60 MT overpull - negat..."


## 7. Strict-vs-fallback diagnostics

These columns show whether the final result came from the strict Excel well or from the task-source `15/9-F-10` fallback.

In [10]:
best_matches[[
    'event_id',
    'well_from_nds',
    'strict_candidate_count',
    'strict_best_pdf',
    'strict_best_score',
    'task_well_candidate_count',
    'task_well_best_pdf',
    'task_well_best_score',
    'final_scope',
    'selection_reason'
]]

,event_id,well_from_nds,strict_candidate_count,strict_best_pdf,strict_best_score,task_well_candidate_count,task_well_best_pdf,task_well_best_score,final_scope,selection_reason
0,1,15/9-F-10,1075,15_9_F_10_2009_04_12.pdf,0.6956,1075,15_9_F_10_2009_04_12.pdf,0.6956,STRICT_WELL,NDS well already equals the task-source well.
1,2,15/9-F-11,211,15_9_F_11_2013_03_13.pdf,0.3656,1075,15_9_F_10_2009_04_15.pdf,0.6752,TASK_WELL_FALLBACK,15/9-F-10 candidate was clearly stronger than the strict NDS-well candidate.
2,3,15/9-F-12,2058,15_9_F_12_2007_07_12.pdf,0.3234,1075,15_9_F_10_2009_04_16.pdf,0.4403,TASK_WELL_FALLBACK,15/9-F-10 candidate was clearly stronger than the strict NDS-well candidate.
3,4,15/9-F-13,0,None,NaN,1075,15_9_F_10_2009_04_18.pdf,0.5896,TASK_WELL_FALLBACK,Strict NDS well was not found in the database; used the task-source well 15/9-F-10.


## 8. Top 3 candidates from each scope

This makes the matching transparent. The reviewer can compare strict well candidates against the `15/9-F-10` candidates.

In [11]:
top_candidates[[
    'event_id',
    'well_from_nds',
    'scope',
    'rank',
    'matched_pdf',
    'matched_well_in_database',
    'operation_start_time',
    'operation_end_time',
    'hybrid_score',
    'critical_token_score',
    'domain_phrase_score',
    'number_overlap_score',
    'matched_excerpt'
]]

,event_id,well_from_nds,scope,rank,matched_pdf,matched_well_in_database,operation_start_time,operation_end_time,hybrid_score,critical_token_score,domain_phrase_score,number_overlap_score,matched_excerpt
0,1,15/9-F-10,STRICT_WELL,1,15_9_F_10_2009_04_12.pdf,15/9-F-10,09:00,11:15,0.6956,0.8667,1.0000,1.0000,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
1,1,15/9-F-10,STRICT_WELL,2,15_9_F_10_2009_04_08.pdf,15/9-F-10,00:00,05:00,0.4731,0.6000,1.0000,0.0000,"drilling -- drill ok Drilled 36"" hole from 165 m to 207 m (17 1/2"" bit depth). 90-100 RPM, 5 ton WOB, 4000-4450 lpm, 3,7 kNm, ROP 5-20 m/hr. Pumped HIVIS pills as per progra m...."
2,1,15/9-F-10,STRICT_WELL,3,15_9_F_10_2009_04_12.pdf,15/9-F-10,07:15,07:30,0.4677,0.6667,1.0000,0.0000,drilling -- drill ok Reamed interval 230-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 97 bar / 40 RPM / Torque 2-4 kNm.
3,1,15/9-F-10,TASK_WELL_FALLBACK,1,15_9_F_10_2009_04_12.pdf,15/9-F-10,09:00,11:15,0.6956,0.8667,1.0000,1.0000,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
4,1,15/9-F-10,TASK_WELL_FALLBACK,2,15_9_F_10_2009_04_08.pdf,15/9-F-10,00:00,05:00,0.4731,0.6000,1.0000,0.0000,"drilling -- drill ok Drilled 36"" hole from 165 m to 207 m (17 1/2"" bit depth). 90-100 RPM, 5 ton WOB, 4000-4450 lpm, 3,7 kNm, ROP 5-20 m/hr. Pumped HIVIS pills as per progra m...."
5,1,15/9-F-10,TASK_WELL_FALLBACK,3,15_9_F_10_2009_04_12.pdf,15/9-F-10,07:15,07:30,0.4677,0.6667,1.0000,0.0000,drilling -- drill ok Reamed interval 230-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 97 bar / 40 RPM / Torque 2-4 kNm.
6,2,15/9-F-11,STRICT_WELL,1,15_9_F_11_2013_03_13.pdf,15/9-F-11,11:00,12:00,0.3656,1.0000,0.0000,0.0000,"drilling -- trip ok Picked up and established parameters. RIH, washing and rotated down from 153 m to TD at 208 m with 1000 lpm, 4 bar, 10 rpm, 1-2 KNm, WOB 1-2 MT tagged botto..."
7,2,15/9-F-11,STRICT_WELL,2,15_9_F_11_2013_03_08.pdf,15/9-F-11,17:00,17:30,0.0515,0.0000,0.0000,0.0000,"drilling -- trip ok Ran in hole with 36"" BHA on 5 1/2"" HWDP and entered F-11 subsea template funnel. RIH and tagged seabed at 143.97 m RKB and spudded well (at 17:25 hrs.)."
8,2,15/9-F-11,STRICT_WELL,3,15_9_F_11_2013_03_08.pdf,15/9-F-11,20:00,23:00,0.0509,0.0000,0.0000,0.0000,"drilling -- drill ok Drilled 36"" hole from 147 m to 160 m. Drilling parameters: 1500 lpm, 11 bar, 40 rpm, 5 KNm, WOB 2-3 MT, Gross ROP 6 m/hr. Pumped and cleaned hole with 10 m..."
9,2,15/9-F-11,TASK_WELL_FALLBACK,1,15_9_F_10_2009_04_15.pdf,15/9-F-10,20:30,00:00,0.6752,1.0000,1.0000,1.0000,"drilling -- trip ok Encounterd thight hole at 958 m MD, took 20-30 MT overpull. Worked string with rotation/flow as limited by torque and overpull. Parameters : Flow 200-800 lp..."


## 9. Method-by-method benchmark

There is no hand-labeled ground truth, so this is a behavioral benchmark. It asks:

> If each algorithm acted alone, which operation row would it choose?

Then it compares each method's top pick with the final hybrid-selected row.

In [12]:
def build_method_benchmark(nds, final_scored_by_event, best_matches):
    rows = []

    for _, nds_row in nds.iterrows():
        event_id = int(nds_row['event_id'])
        scored = final_scored_by_event.get(event_id)
        final_row = best_matches[best_matches['event_id'] == event_id]

        if scored is None or scored.empty or final_row.empty:
            rows.append({
                'event_id': event_id,
                'well_from_nds': nds_row['Well'],
                'event': nds_row['Event'],
                'final_scope': None,
                'method': 'NO CANDIDATES',
                'top_method_score': np.nan,
                'method_top_pdf': None,
                'method_top_operation_id': None,
                'method_top_hybrid_score': np.nan,
                'agrees_with_final_hybrid_best': False,
                'method_top_excerpt': None,
            })
            continue

        final_operation_id = int(final_row['operation_id'].iloc[0])
        final_scope = final_row['final_scope'].iloc[0]

        for score_col in SCORE_COLUMNS:
            best_i = int(scored[score_col].idxmax())
            best = scored.loc[best_i]

            rows.append({
                'event_id': event_id,
                'well_from_nds': nds_row['Well'],
                'event': nds_row['Event'],
                'final_scope': final_scope,
                'method': METHOD_LABELS[score_col],
                'top_method_score': round(float(best[score_col]), 4),
                'method_top_pdf': best['filename'],
                'method_top_operation_id': int(best['operation_id']),
                'method_top_hybrid_score': round(float(best['hybrid_score']), 4),
                'agrees_with_final_hybrid_best': bool(best['operation_id'] == final_operation_id),
                'method_top_excerpt': best['candidate_text'],
            })

    return pd.DataFrame(rows)


method_benchmark = build_method_benchmark(nds, final_scored_by_event, best_matches)

method_benchmark[[
    'event_id',
    'well_from_nds',
    'final_scope',
    'method',
    'top_method_score',
    'method_top_pdf',
    'method_top_hybrid_score',
    'agrees_with_final_hybrid_best',
    'method_top_excerpt'
]]

,event_id,well_from_nds,final_scope,method,top_method_score,method_top_pdf,method_top_hybrid_score,agrees_with_final_hybrid_best,method_top_excerpt
0,1,15/9-F-10,STRICT_WELL,Word TF-IDF,0.3227,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
1,1,15/9-F-10,STRICT_WELL,Character TF-IDF,0.5336,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
2,1,15/9-F-10,STRICT_WELL,Critical token score,0.8667,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
3,1,15/9-F-10,STRICT_WELL,Domain phrase score,1.0000,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
4,1,15/9-F-10,STRICT_WELL,Number overlap,1.0000,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
5,1,15/9-F-10,STRICT_WELL,Token overlap,0.5000,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
6,1,15/9-F-10,STRICT_WELL,Fuzzy similarity,0.4260,15_9_F_10_2009_06_11.pdf,0.0242,False,"interruption -- maintain ok Serviced TDS, inspected for loose items prior to pulling casing."
7,1,15/9-F-10,STRICT_WELL,Hybrid final score,0.6956,15_9_F_10_2009_04_12.pdf,0.6956,True,drilling -- drill ok Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pil...
8,2,15/9-F-11,TASK_WELL_FALLBACK,Word TF-IDF,0.1655,15_9_F_10_2009_04_15.pdf,0.6752,True,"drilling -- trip ok Encounterd thight hole at 958 m MD, took 20-30 MT overpull. Worked string with rotation/flow as limited by torque and overpull. Parameters : Flow 200-800 lp..."
9,2,15/9-F-11,TASK_WELL_FALLBACK,Character TF-IDF,0.3507,15_9_F_10_2009_05_14.pdf,0.4961,False,"drilling -- trip ok POOH with 12 1/4"" BHA on 5 1/2"" DP from 3243 m to 2907 m. Encountered tight spot at 2907 m, max OP 20 ton."


## 10. Benchmark summaries

In [13]:
matched_only = best_matches[best_matches['status'].str.startswith('MATCHED', na=False)].copy()
selected_match_score_summary = matched_only[SCORE_COLUMNS].describe().T.round(4)
selected_match_score_summary

,count,mean,std,min,25%,50%,75%,max
word_tfidf_score,4.0,0.1993,0.1025,0.0798,0.1441,0.1973,0.2525,0.3227
char_tfidf_score,4.0,0.3416,0.1501,0.1711,0.2706,0.3308,0.4017,0.5336
critical_token_score,4.0,0.8420,0.1224,0.7143,0.7688,0.8268,0.9000,1.0000
domain_phrase_score,4.0,0.8393,0.2360,0.5000,0.7678,0.9286,1.0000,1.0000
number_overlap_score,4.0,0.9167,0.1667,0.6667,0.9167,1.0000,1.0000,1.0000
token_overlap_score,4.0,0.5329,0.2473,0.2000,0.4250,0.5834,0.6912,0.7647
fuzzy_score,4.0,0.1284,0.0706,0.0394,0.1055,0.1312,0.1541,0.2120
hybrid_score,4.0,0.6002,0.1161,0.4403,0.5523,0.6324,0.6803,0.6956


In [14]:
method_benchmark_matched = method_benchmark[method_benchmark['method'] != 'NO CANDIDATES'].copy()

method_summary = (
    method_benchmark_matched
    .groupby('method')
    .agg(
        event_count=('event_id', 'count'),
        avg_top_method_score=('top_method_score', 'mean'),
        median_top_method_score=('top_method_score', 'median'),
        min_top_method_score=('top_method_score', 'min'),
        max_top_method_score=('top_method_score', 'max'),
        agreement_rate_with_hybrid=('agrees_with_final_hybrid_best', 'mean'),
    )
    .reset_index()
)

num_cols = [
    'avg_top_method_score',
    'median_top_method_score',
    'min_top_method_score',
    'max_top_method_score',
    'agreement_rate_with_hybrid',
]
method_summary[num_cols] = method_summary[num_cols].round(4)

method_summary.sort_values('agreement_rate_with_hybrid', ascending=False)

,method,event_count,avg_top_method_score,median_top_method_score,min_top_method_score,max_top_method_score,agreement_rate_with_hybrid
1,Critical token score,4,0.8420,0.8268,0.7143,1.0000,1.00
2,Domain phrase score,4,0.8393,0.9286,0.5000,1.0000,1.00
4,Hybrid final score,4,0.6002,0.6324,0.4403,0.6956,1.00
5,Number overlap,4,1.0000,1.0000,1.0000,1.0000,0.75
6,Token overlap,4,0.6328,0.6334,0.5000,0.7647,0.75
7,Word TF-IDF,4,0.2332,0.2222,0.1655,0.3227,0.75
0,Character TF-IDF,4,0.3655,0.3542,0.2199,0.5336,0.50
3,Fuzzy similarity,4,0.4613,0.4352,0.4048,0.5699,0.00


## 11. Export Excel review file

The Excel workbook includes:

- `best_matches` — final one-row-per-event result
- `top_candidates` — strict and fallback top candidates
- `method_benchmark` — method-by-method comparison
- `method_summary` — benchmark summary
- `score_summary` — score distribution for selected matches
- `candidate_counts` — operation row count by well
- `nds_events` — original NDS inputs

In [ ]:
candidate_counts = (
    candidates.groupby('wellbore_id')
    .size()
    .reset_index(name='operation_count')
    .sort_values('wellbore_id')
)

OUTPUT_PATH = SQLITE_PATH.parent / 'task2_nds_event_matches.xlsx'

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    best_matches.to_excel(writer, sheet_name='best_matches', index=False)
    top_candidates.to_excel(writer, sheet_name='top_candidates', index=False)
    method_benchmark.to_excel(writer, sheet_name='method_benchmark', index=False)
    method_summary.to_excel(writer, sheet_name='method_summary', index=False)
    selected_match_score_summary.to_excel(writer, sheet_name='score_summary')
    candidate_counts.to_excel(writer, sheet_name='candidate_counts', index=False)
    nds.to_excel(writer, sheet_name='nds_events', index=False)

print('Saved:', OUTPUT_PATH.resolve())

Saved: /mnt/data/task2_nds_event_matches_fixed.xlsx
